# Eksperimen 3: IndoBERT Fine-Tuned (Referensi dan Jawaban)

**GEMASTIK KTI 2026** | Tim Peneliti

Format masukan model: `[CLS] kunci_jawaban [SEP] jawaban_siswa [SEP]`

Pendekatan ini memberikan model bahasa akses penuh ke kunci jawaban secara bersamaan. Hal ini memastikan model dievaluasi pada kondisi yang setara dengan model fitur leksikal, di mana keduanya bertugas mengomparasikan kedua teks secara langsung.

## 0. Persiapan Lingkungan dan Konfigurasi

In [ ]:
import sys
import os

try:
    import google.colab
    IN_COLAB = True
    print("Lingkungan Eksekusi: Google Colab")
    if not os.path.exists("/content/indo-asag"):
        os.system("git clone https://github.com/Rnov24/indo-asag.git /content/indo-asag")
    os.system("pip install -q -e /content/indo-asag[all]")
    REPO_ROOT = "/content/indo-asag"
except ImportError:
    IN_COLAB = False
    try:
        REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
    except NameError:
        REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print(f"Lingkungan Eksekusi: Lokal ({REPO_ROOT})")

sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from indo_asag.data import load_dataset
from indo_asag.evaluation import run_multi_seed
from indo_asag.utils import set_seed, load_config
from indo_asag.utils.github import auto_push_to_github

config = load_config(os.path.join(REPO_ROOT, "configs", "base.yaml"))
SEEDS = config["seeds"]
RESULTS_DIR = os.path.join(REPO_ROOT, "results", "prelim")
PREDS_DIR = os.path.join(RESULTS_DIR, "predictions")
CHECKPOINT_DIR = os.path.join(REPO_ROOT, "checkpoints")

## 1. Pemuatan Dataset

In [ ]:
DATA_PATH = os.path.join(REPO_ROOT, config["data"]["path"])
df = load_dataset(DATA_PATH)

## 2. Eksekusi Eksperimen 3

In [ ]:
from indo_asag.models.bert_regressor import BertRegressor

print("\n" + "=" * 60)
print("EXP 3: IndoBERT Fine-Tuned (Referensi dan Jawaban)")
print("=" * 60)

bert = BertRegressor(
    model_name=config["model"]["bert"]["name"],
    dropout=config["model"]["bert"]["dropout"],
)

bert_oof_preds = {s: np.zeros(len(df)) for s in SEEDS}
bert_oof_embs = {s: np.zeros((len(df), 768)) for s in SEEDS}

def exp3_predict(train_df, val_df, fold, seed=42):
    preds, embs = bert.train_fold(
        train_df, val_df, fold,
        text_a_col="reference_answer",
        text_b_col="student_answer",
        epochs=config["model"]["bert"]["epochs"],
        batch_size=config["model"]["bert"]["batch_size"],
        lr=config["model"]["bert"]["learning_rate"],
        save_path=os.path.join(CHECKPOINT_DIR, f"indobert_seed{seed}_fold{fold}.pt")
    )
    bert_oof_preds[seed][val_df.index] = preds
    bert_oof_embs[seed][val_df.index] = embs
    return preds

exp3_summary = run_multi_seed(df, exp3_predict, seeds=SEEDS)
print(f"  QWK: {exp3_summary['QWK']}, Pearson: {exp3_summary['Pearson']}")

## 3. Penyimpanan Prediksi dan Embeddings (Out-of-Fold)

In [ ]:
print("\nMenyimpan array prediksi dan embeddings OOF ke disk...")
for s in SEEDS:
    np.save(os.path.join(PREDS_DIR, f"exp3_indobert_oof_seed{s}.npy"), bert_oof_preds[s])
    np.save(os.path.join(PREDS_DIR, f"exp3_indobert_emb_seed{s}.npy"), bert_oof_embs[s])
print("[OK] Prediksi dan embeddings berhasil disimpan.")

## 4. Publikasi Otomatis ke GitHub

In [ ]:
auto_push_to_github("chore(auto): menyimpan prediksi Eksperimen 3 IndoBERT", IN_COLAB, REPO_ROOT)